In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import geopandas as gpd
from shapely import affinity, Polygon, Point, LineString
import folium
from folium import plugins
from folium.plugins import HeatMap
import osmnx as ox
from math import radians, cos, sin, asin, sqrt
import requests
import urllib
import networkx as nx
import pickle
from scipy import stats

Image.MAX_IMAGE_PIXELS = None

import branca.colormap as cm

## Problem Formulation

The first step of this project is to come up with a graph showing potential spreading vectors within a forest.  We assume that a fire can start anywhere in the forest and that the likelyhood any area will have fire is proportional to the how many different ways fire could spread to an area. To quantify this, we formalized an approach by which we can represent fire propagation risk as a network of spatially related entities. This network should mimic other spreading networks, such as infection networks. Infection networks are often constructed from social networks, where nodes represent people or potential hosts of an infection and edges connect people who are exposed to each other, creating a potential opportunity for the spread of disease. Similarly, for the case of a fire spread network, the nodes represent the locations which fire can be present, and edges connect the locations between which fires can spread. Calculating the priority map has been broken up into the following steps:

* A. Define Active Area (ROI)
* B. Define Network Nodes
* C. Find Network Edges
* D. Add Elevation to Nodes, Save and Plot Elevation

## A. Define Active Area (rectangle that we will be considering)

To develop such a network, we first determine the nodes. This starts by defining a region of interest (ROI), which we defined to be between [121.75° West – 122° West] and [37° North – 37.25° North]. The forested areas in that ROI are then identified by querying OpenStreetMaps. 

In [ ]:
active_zone_bounds =  ((-122.00, -121.75),(37.00, 37.25))  ## (latitude range, longitude range)

active_zone_bounds

In [ ]:
def get_poly_sqr_from_bounds(active_zone_bounds):
    
    active_zone_ll = Polygon([[active_zone_bounds[0][0],active_zone_bounds[1][0]],
                              [active_zone_bounds[0][1],active_zone_bounds[1][0]],
                              [active_zone_bounds[0][1],active_zone_bounds[1][1]],
                              [active_zone_bounds[0][0],active_zone_bounds[1][1]],
                              [active_zone_bounds[0][0],active_zone_bounds[1][0]]])

    return active_zone_ll


In [ ]:
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)

lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 9

# Basemap
m = folium.Map([lat, lon], tiles='OpenStreetMap', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

m

In [ ]:
m.save("../figures/roi.png")

## B. Define Network Nodes

The forested areas are defined to be regions with either the ‘landuse’ tag set to ‘forest’ or the ‘natural’ tag set to ‘wood’. To turn these areas into nodes, a grid of 51 by 63 points is defined over the area of interest. This grid size was selected so the x and y spacing between points is approximately equal and so that the network produced is densely, but not overly, connected. Grid points which lie within the forested areas are then kept for consideration as network nodes.

In [ ]:
def haversine(lon1, lat1, lon2, lat2, r_type = 'miles'):
    """
    Calculate the great circle distance in kilometers between two points 
    on the earth (specified in decimal degrees)
    """
    # convert decimal degrees to radians 
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula 
    dlon = lon2 - lon1 
    dlat = lat2 - lat1 
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a)) 
    
    if r_type == 'miles':    
        r = 3956 #6371 # Radius of earth in miles. Determines return value units.
    elif r_type == 'kilometers':
        r = 6371 # Radius of earth in kilometers. Determines return value units.
        
    return c * r

def get_network_node_locations(active_zone_bounds, div_factor, geometry_type = 'grid'):
    active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)

    tree_polygons = ox.features.features_from_polygon(active_zone_ll, tags = {'landuse': ['forest'], 'natural': ['wood']}) # land use tags used to define forest
    
    tree_polygons.reset_index(inplace = True)
    
    if geometry_type == 'grid':
        
        points = []
        x_values = []
        y_values = []
        
        x_points = int(100*(active_zone_bounds[0][1] - active_zone_bounds[0][0])/div_factor) 
        xy_factor = haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][0], active_zone_bounds[1][1])/haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][1], active_zone_bounds[1][0]) 
    
        for x in range(x_points + 1):
            for y in range(int(x_points*xy_factor) + 1):
                point = Point(active_zone_bounds[0][0] + x/x_points*(active_zone_bounds[0][1] - active_zone_bounds[0][0]),
                              active_zone_bounds[1][0] + y/int(x_points*xy_factor)*(active_zone_bounds[0][1] - active_zone_bounds[0][0]))
    
                if np.sum(tree_polygons.apply(lambda row:row['geometry'].intersects(point), axis=1)) > 0:
                    points.append(point)
                    x_values.append(x)
                    y_values.append(y)
    
        geometries = pd.DataFrame(points, columns=['geometry'])
    
    return geometries, tree_polygons, x_values, y_values


def plot_network_nodes(active_zone_ll, geometries, zoom_start_val = 11):

    lon, lat = active_zone_ll.centroid.coords[0]
    min_area = 1e-6
    polygons = []
    centroids = []
    
    # Basemap
    m = folium.Map([lat, lon], tiles='OpenStreetMap', zoom_start=zoom_start_val)#, height=500)#, tiles='OpenStreetMap')
    
    # Bounding box around active zone = Red
    sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
    geo_j.add_to(m)
    
    # Plot OSM Results
    for tree_geom in geometries['geometry']:
    
        # Plot Centroids
        if tree_geom.geom_type == 'Polygon':
            centroid = tree_geom.centroid
        elif tree_geom.geom_type == 'Point':
            centroid = tree_geom
    
        folium.Circle(location=[centroid.y, centroid.x],
                                radius=0.4,
                                color = 'green').add_to(m)#weight=5, color = 'green').add_to(m)
    
        centroids.append(centroid)

    return centroids, m

In [ ]:
geometry_type = 'grid' # grid or polygons

tags = {'landuse': ['forest'], 'natural': ['wood']} #
div_factor = 0.5 #0.6 #0.9

x_distance = haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][1], active_zone_bounds[1][0])
y_distance = haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][0], active_zone_bounds[1][1])

geometries, tree_polygons, x_values, y_values = get_network_node_locations(active_zone_bounds, div_factor, geometry_type = 'grid')

len(geometries), geometries.head(2), geometries.tail(2)


In [ ]:
x_points = int(100*(active_zone_bounds[0][1] - active_zone_bounds[0][0])/div_factor) 
xy_factor = haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][0], active_zone_bounds[1][1])/haversine(active_zone_bounds[0][0], active_zone_bounds[1][0], active_zone_bounds[0][1], active_zone_bounds[1][0]) 


In [ ]:
print('number of points in x: {}, number of points in y: {}'.format(x_points + 1, int(x_points*xy_factor) + 1))

In [ ]:
geometries.to_csv('../data/tree_nodes.csv', index=False)

In [ ]:
centroids, m = plot_network_nodes(active_zone_ll, geometries, zoom_start_val = 11)

m

In [ ]:
m.save("../figures/forest_graph_nodes.png")

## C. Find Network Edges

### C.1 Get the Elevation of the Nodes in the Network

The elevation of each point was queried via the Elevation Point Query Service and stored as a variable to be used in the edge creation steps.

In [ ]:
## optionally you can save centroids and then re-open them here

# with open('fire_centroids_2.pkl', 'rb') as handle:
#     centroids = pickle.load(handle)

In [ ]:
def elevation_function(df, url, lat_column, lon_column):
    """Query service using lat, lon. add the elevation values as a new column."""
    elevations = []
    counter = 0
    for lat, lon in zip(df[lat_column], df[lon_column]):
        if counter % 100 == 0:
            print(counter)
        # define rest query params
        params = {
            'output': 'json',
            'x': lon,
            'y': lat,
            'units': 'Meters'}
        
        try:
            result = requests.get((url + urllib.parse.urlencode(params)))
            elevations.append(result.json()['value'])
        except Exception as e:
            print(f"Error for (lat, lon) ({lat}, {lon}):",e)
            elevations.append(-500)
        counter += 1

    df['elev_meters'] = elevations
    return df
    

# USGS Elevation Point Query Service
url = r'https://epqs.nationalmap.gov/v1/json?'

# coordinates with known elevation 
lat = [centroid.y for centroid in centroids]
lon = [centroid.x for centroid in centroids]

# create data frame
elevation_df = pd.DataFrame({
    'lat': lat,
    'lon': lon
})

elevation_df = elevation_function(elevation_df, url, 'lat', 'lon')
elevation_df['elev_meters'] = elevation_df.elev_meters.astype(float)
elevation_df.head()


In [ ]:
# elevation_df.to_csv('elevations_grid_2.csv')
# elevation_df = pd.read_csv('elevations_grid_2.csv')
elevation_df.to_csv('../data/elevations_grid_3.csv')

In [ ]:
elevation_df.head()

### C.2 Add Network Edges

Next, we define the network edges. There are several ways in which a fire spreads, including spreading, short-range spotting, and long-range spotting.  Spotting describes when embers or firebrands are carried away from the original source of the fire, and land on additional fuel to start a new ‘spot’ fire. Long-range spotting occurs when the firebrands are lofted into the air and travel a significant distance before landing. The maximum on-ground distance those firebrands can travel, while remaining lit to be able to start new fires, is called the maximum spotting distance. Short-range spotting describes spotting at distances less than the maximum spotting distance and is often omitted from fire modeling on the assumption that this type of spotting is typically accounted for when modeling fire spreading. Therefore, only long-range spotting and spreading are accounted for in the creation of this network. For simplicity, these were the only factors considered in modeling the fire propagation for this paper. It should be noted that real fire propagation is influenced by the previously mentioned factors, such as environment and terrain, which could be incorporated into the edge creation process described below.

To account for spreading, we consider that two points which have a line connecting them in the original gridded pattern within a forested region as described above can have direct fire spreading between them. These points are therefore connected via an edge in our network structure. To account for long-range spotting, an edge is also created between two points if the distance between those points is less than the maximum spotting distance. The maximum spotting distance is given by:

```s=(-1.253 x〖 10〗^(-4)  w+3.304 x 10^(-5) )*(t+e)+0.4297w+0.01065 ```

where s is the maximum fire spotting distance in miles, w is the wind speed in mph, t is the tree height in feet, and e is the elevation difference between the points in feet. This equation was derived from Figure “NOM 4. Maximum Spotting Distance” from the NWCG . In this equation, it is assumed that the average tree height is 50 feet and that there is 40 mph wind which can be coming from any direction. In addition to simplifying the network creation process, accounting for all wind directions aligns with the worst-case scenario where no prior information about the fire or wind is available.  This resulted inThe resulting network contained multiple small isolated subgraphs, e.g. the cluster of tree nodes in the southwest corner of Figure 4. These small isolated subgraphs would not be impacted by fire spreading in the main body of the network but would affect the optimization while it attempts to create near-equal partitions. Therefore, we removed the isolated subgraphs, resulting in a network with a total of 1492 nodes and a total of 4,637 edges. 

Resource: https://www.nwcg.gov/publications/pms437/crown-fire/spotting-fire-behavior
(max spotting distance comes from NOM. 4)


In [ ]:
## These numbers come from a figure at the resource we used to get the fire spreading

elev_to_max_dist = {40: {25: 1.73, 50: 1.42, 100: 1.11, 200: 0.79},# wind speed: {effective tree height: max distance}
                    30: {25: 1.29, 50: 1.07, 100: 0.83, 200: 0.60},
                    20: {25: 0.87, 50: 0.72, 100: 0.57, 200: 0.42},
                    10: {25: 0.44, 50: 0.36, 100: 0.29, 200: 0.20}}

def max_spotting_distance(wind_speed, difference_in_elevation):
    slope, intercept, r_value, p_value, std_err = stats.linregress(list(elev_to_max_dist[wind_speed].keys()),
                                                                   list(elev_to_max_dist[wind_speed].values()))

    difference_in_elevation = 3.28084*difference_in_elevation # convert from meters to feet
    
    return slope*(difference_in_elevation) + intercept # returns results in miles
    ## this should result in the following equation: s=(-1.253 x〖 10〗^(-4)  w+3.304 x 10^(-5) )*(t+e)+0.4297w+0.01065


In [ ]:
def add_centroid(geometries, tree_num1):
    if type(geometries['geometry'][tree_num1]) == Point:
        lon = geometries['geometry'][tree_num1].coords.xy[0][0]
        lat = geometries['geometry'][tree_num1].coords.xy[1][0]
    else:
        lon = geometries['geometry'][tree_num1].centroid.coords.xy[0][0]
        lat = geometries['geometry'][tree_num1].centroid.coords.xy[1][0]

    return lon, lat

G = nx.Graph()
#cutoff_distance = 1.0 # miles
wind_speed = 20
adjacent_dist = np.sqrt(((active_zone_bounds[0][1] - active_zone_bounds[0][0])/50)**2 + ((active_zone_bounds[1][1] - active_zone_bounds[1][0])/62)**2)

for tree_num1, tree_geom in enumerate(geometries['geometry']):
    if tree_num1 % 100 == 0:
        print(tree_num1)
        
    lon1, lat1 = add_centroid(geometries, tree_num1)
    G.add_node(tree_num1, pos=(lon1, lat1))

    for tree_num2 in range(0,tree_num1): # for all previous trees
        edge_exists = False
        lon2, lat2 = add_centroid(geometries, tree_num2)

        ## Tree Locations Connected by Spreading
        ##      - Secifically if line between points is within tree polygons, then it can spread by contact/proximity
        ##      - Could change to edge of polygons instead of centroids -> higher connectivity
        line_of_sight = LineString([Point(lon1, lat1), Point(lon2, lat2)])
        if np.abs(line_of_sight.length) <= adjacent_dist+0.001:
            for tree_polygon in tree_polygons['geometry']:
                if tree_polygon.contains(line_of_sight):
                    G.add_edge(tree_num1, tree_num2)
                    edge_exists = True
        
        ## Get Cutoff for Long-Range Spreading Edge
        distance = haversine(lon1, lat1, lon2, lat2)
        difference_in_elevation = abs(elevation_df.loc[tree_num1, 'elev_meters'] - elevation_df.loc[tree_num2, 'elev_meters']) + 15.24 # effective elevation = elevation + tree height, tree height = 50 ft.
        cutoff_distance = max_spotting_distance(wind_speed, difference_in_elevation)

        ## Check if Points are withing by Long Range Spotting Distance
        if edge_exists == False:
            if distance < cutoff_distance:
                G.add_edge(tree_num1, tree_num2)


In [ ]:
for tree_num1, tree_geom_1 in enumerate(geometries['geometry'][165:166]):
    for tree_num1, tree_geom_2 in enumerate(geometries['geometry']):
        line_of_sight = LineString([Point(tree_geom_1.x, tree_geom_1.y), Point(tree_geom_2.x, tree_geom_2.y)])
        
        if np.abs(line_of_sight.length) <= adjacent_dist+0.001:
            print(tree_geom_1, tree_geom_2)

In [ ]:
line_of_sight.length

In [ ]:
len(list(G.nodes)), len(list(G.edges))

In [ ]:
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 9
min_area = 1e-6

# Basemap
m = folium.Map([lat, lon], tiles='OpenStreetMap', zoom_start=zoom_start_val)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

# Plot OSM Results
for tree_geom in geometries['geometry']:

    if tree_geom.area > min_area:
        if tree_geom.geom_type == 'Polygon':
            centroid = tree_geom.centroid
        elif tree_geom.geom_type == 'Point':
            centroid = tree_geom

        folium.CircleMarker(location=[centroid.y, centroid.x],
                            radius=0.1,
                            color = 'green').add_to(m)  #weight=5, color = 'green').add_to(m)

# Plot edges
for edge in G.edges():
    c1 = centroids[edge[0]]
    c2 = centroids[edge[1]]
    folium.PolyLine([[c1.y,c1.x],[c2.y,c2.x]], weight=1).add_to(m)

m

In [ ]:
m.save("../figures/forest_graph_edges.png")

In [ ]:
elevation_df

## D. Add Elevation to Nodes, Save and Plot Elevation

This section is just plotting the elevation of nodes and saving the final graph (fire_graph_3.pkl). No additional analysis is done here.

In [ ]:
len(G.nodes)

In [ ]:
nx.set_node_attributes(G, list(elevation_df['elev_meters']), "elevation_meters")

In [ ]:
elevations = {}

for node_num, node_elevation in enumerate(list(elevation_df['elev_meters'])):
    elevations.update({node_num:node_elevation})

nx.set_node_attributes(G, elevations, "elevation_meters")

In [ ]:
G.nodes[0]

In [ ]:
# pickle.dump(G, open('../data/fire_graph.pkl', 'wb'))
# pickle.dump(G, open('fire_graph_2.pkl', 'wb'))
pickle.dump(G, open('../data/fire_graph_3.pkl', 'wb'))

In [ ]:
# with open('fire_centroids_2.pkl', 'wb') as handle:
#     pickle.dump(centroids, handle)
with open('../data/fire_centroids_3.pkl', 'wb') as handle:
    pickle.dump(centroids, handle)

In [ ]:
active_zone_ll = get_poly_sqr_from_bounds(active_zone_bounds)
lon, lat = active_zone_ll.centroid.coords[0]
zoom_start_val = 11
min_area = 1e-6
polygons = []
# centroids = []

# Basemap
m = folium.Map([lat, lon], tiles='OpenStreetMap', zoom_start=zoom_start_val)#, height=500)#, tiles='OpenStreetMap')

# Bounding box around active zone = Red
sim_geo = gpd.GeoSeries(active_zone_ll).simplify(tolerance=0.001)
geo_j = sim_geo.to_json()
geo_j = folium.GeoJson(data=geo_j, style_function=lambda x: {"fill":False,"color": "red"})#orange"})
geo_j.add_to(m)

# Plot edges
for edge in G.edges():
    c1 = centroids[edge[0]]
    c2 = centroids[edge[1]]
    folium.PolyLine([[c1.y,c1.x],[c2.y,c2.x]], weight=1).add_to(m)

# Plot OSM Results
colormap = cm.LinearColormap(colors=['black','purple','red','yellow'],vmin=elevation_df['elev_meters'].min(),vmax=elevation_df['elev_meters'].max(), caption="Hubness")
i = 0
for centroid in centroids:
    
    folium.Circle(location=[centroid.y, centroid.x],
                            radius=0.4,
                            color = colormap(elevation_df.iloc[i]['elev_meters'])).add_to(m)#weight=5, color = 'green').add_to(m)
    i += 1

    # centroids.append(centroid)

m

In [ ]:
m.save("../figures/forest_graph_nodes_by_priority.png")

In [ ]:
print('number of points in x: {}, number of points in y: {}'.format(x_points + 1, int(x_points*xy_factor) + 1))

In [ ]:
priority_map = np.zeros((int(x_points*xy_factor) + 1, x_points+1))

In [ ]:
geometries['x_value'] = x_values
geometries['y_value'] = y_values

In [ ]:
for node in G.nodes():
    x_value = int(geometries[geometries['geometry'] == Point(G.nodes()[node]['pos'])]['x_value'].values[0])
    y_value = int(geometries[geometries['geometry'] == Point(G.nodes()[node]['pos'])]['y_value'].values[0])

    node_index = int(geometries[geometries['geometry'] == Point(G.nodes()[node]['pos'])].index[0])    
    n_edges = len(G.edges(node_index))
    priority_map[y_value, x_value] = n_edges


In [ ]:
f, ax = plt.subplots(1, 1)

plt.imshow( priority_map, origin='lower')
cbar = plt.colorbar()

cbar.set_label('Spreading Potential', labelpad=10, rotation = -90)

f.savefig('../figures/priority_map/priority_map.png', bbox_inches='tight')